# Method: Graph community
The notebook comprises two parts. The first part creates the graph and determines the communities (clusters) according to the Louvain approach. The second part uses the `communities.json` file generated in the first part to retrieve the associated articles from a query, according to the algorithm depicted in the report.

## Data Loading - Graph Creation

In [23]:
import networkx as nx
from networkx.algorithms.community import louvain_communities
from networkx.algorithms.community.quality import modularity
import json
import matplotlib.pyplot as plt
from operator import itemgetter
from typing import Set, Dict, Any, Tuple
import os
import numpy as np

from operator import itemgetter # Utilisé pour trier

`FILTERED_JSON_PATH` allows you to select the subsampled dataset you want to use. This datasets are located in the folder `data/processed`.

In [3]:
# --- Constants for file paths ---
FILTERED_JSON_PATH = 'data/processed/filtered_articles_stratified.json'


def create_graph_from_filtered_json(
    json_path: str = FILTERED_JSON_PATH,
) -> nx.DiGraph:
    """
    Builds a directed graph (DiGraph) from the filtered JSON file.
    The graph is created in memory and is NOT saved or loaded from disk.
    """

    print("--- Creating graph from filtered JSON (in memory)... ---")
    
    G = nx.DiGraph()
    
    # 1. Load JSON data (Consolidated try/except)
    try:
        with open(json_path, 'r', encoding='utf-8') as json_file:
            data = json.load(json_file)
            
        articles = data.get('articles', [])
        
        # 2. Graph creation
        for article in articles:
            article_id = article.get('id')
            if article_id is not None:
                G.add_node(article_id) 
                for link in article.get('refs', []):
                    G.add_edge(article_id, link)
                    
             
    except FileNotFoundError:
        print(f"Error: Filtered JSON file not found at: {json_path}. Run 'filter_json_and_save' first.")
        return nx.DiGraph()
    except json.JSONDecodeError:
        print(f"Error: Invalid JSON format in file: {json_path}")
        return nx.DiGraph()
    except Exception as e:
        print(f"Error during graph creation: {e}")
        return nx.DiGraph()

    print(f"Graph created: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.")
    
    return G

## Community generation

After community detection, a JSON file called `communities.json` is saved in the `data/processed` folder. This file contains all the Louvain clusters, along with the representative for each one, which is simply the most cited article in that cluster.

In [ ]:
def community_detection():
    results = [] 
    G = create_graph_from_filtered_json()


    # Louvain community detection
    communities = louvain_communities(G, seed=42)

    for comm in communities:
        # Identify the node with the highest degree as the representative
        # representative_article = sorted(comm, key=lambda x: G.in_degree(x), reverse=True)[0]
        results.append({
            # 'representative_node' : representative_article,
            # # Convert set to list for JSON serialization
            'community' : sorted(comm, key=lambda x: G.in_degree(x), reverse=True)
        })

    with open('data/processed/communities.json', 'w') as outfile:
        # Use indent for better JSON readability
        json.dump(results, outfile, indent=4) 

if __name__ == '__main__':
    community_detection()

--- Creating graph from filtered JSON (in memory)... ---
Graph created: 49919 nodes, 116528 edges.


In [11]:
file_path = 'data/processed/communities.json' 

with open(file_path, 'r') as infile:
    commu = json.load(infile)
print(len(commu))

article_name = {}
number_citation = {}
with open(FILTERED_JSON_PATH, 'r' ) as infile:
    nodes = json.load(infile)
for article in nodes['articles']:
    article_name[article['id']] = article['title']

20708


In [ ]:
print('Number of clusters : ', len(commu))
print('Average number of article per cluster : ', np.mean([len(l['community']) for l in commu]))

print('---------')
print('LIST OF PROCESSED COMMUNITIES AND THEIR REPRESENTATIVE NODE')
print('---------')


for i,commu_i in enumerate(commu):
    print(f"Commu numéro {i+1} : {article_name[commu_i['community'][0]]}")


Number of clusters :  20708
Average number of article per cluster :  2.4106142553602474
---------
LIST OF PROCESSED COMMUNITIES AND THEIR REPRESENTATIVE NODE
---------
Commu numéro 1 : A M\"obius Characterization of Metric Spheres
Commu numéro 2 : Variational Inference with Mixtures of Isotropic Gaussians
Commu numéro 3 : Thin Set Versions of Hindman's Theorem
Commu numéro 4 : Center-to-limb polarization in continuum spectra of F, G, K stars
Commu numéro 5 : Coulomb correlations of a few body system of spatially separated charges
Commu numéro 6 : A model comparison of 2D Cartesian and 2D axisymmetric models for
  positive streamer discharges in air
Commu numéro 7 : Behavioural investors in conic market models
Commu numéro 8 : In-memory eigenvector computation in time O(1)
Commu numéro 9 : Microstructure and velocity fluctuations in sheared suspensions
Commu numéro 10 : Approach of the constitutive material behaviour of textile composites
  through simulation
Commu numéro 11 : Optimizin

## Recommandation

Once the communities have been processed, the closest article to the query is found using a retrieval method (here embedding method), and the representative article of the community in which the node participates is associated with it.

In [13]:
# Core Python libraries
import numpy as np
import pandas as pd
from tqdm import tqdm
import json
import re
import os

# NLP and Embeddings
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
import faiss


# Visualization and evaluation
import matplotlib.pyplot as plt
import seaborn as sns



Fist, the embedding is created for the subset.

In [14]:
def compute_and_save_embeddings(texts, doc_ids, model_name='all-mpnet-base-v2',
                                batch_size=64, out_emb_path='embeddings.npy',
                                out_ids_path='doc_ids.json', overwrite=True):
    if os.path.exists(out_emb_path) and not overwrite:
        raise FileExistsError(f"{out_emb_path} already exists. Set overwrite=True to replace.")

    model = SentenceTransformer(model_name)
    n = len(texts)
    emb_dim = model.get_sentence_embedding_dimension()

    emb_memmap = np.lib.format.open_memmap(out_emb_path, mode='w+', dtype='float32', shape=(n, emb_dim))

    for i in tqdm(range(0, n, batch_size), desc="Embedding batches"):
        batch_texts = texts[i:i+batch_size]
        batch_emb = model.encode(batch_texts, show_progress_bar=False, convert_to_numpy=True)
        # normalize rows to unit vectors (for cosine via inner product)
        norms = np.linalg.norm(batch_emb, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        batch_emb = batch_emb / norms
        emb_memmap[i:i+len(batch_emb)] = batch_emb.astype('float32')

    # ensure data flushed to disk
    del emb_memmap

    # save doc ids as JSON
    with open(out_ids_path, 'w', encoding='utf-8') as f:
        json.dump(list(doc_ids), f, ensure_ascii=False)

    print(f"Saved embeddings -> {out_emb_path}")
    print(f"Saved doc ids -> {out_ids_path}")

In [ ]:
from sentence_transformers import SentenceTransformer

# Define output paths variables to ensure consistency
output_emb_file = 'data/processed/embeddings.npy'
output_ids_file = 'data/processed/doc_ids.json'

# Check if the output files already exist
if os.path.exists(output_emb_file) and os.path.exists(output_ids_file):
    # Files exist: Skip the expensive computation
    print(f"Embeddings and IDs found at '{output_emb_file}'. Skipping computation.")
else:
    # Files do not exist: Proceed with loading data and computing embeddings
    print("Output files not found. Starting data loading and embedding computation...")

    with open(FILTERED_JSON_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)

    texts = [article["clean_text"] for article in data["articles"]]
    doc_ids = [article["id"] for article in data["articles"]]

    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    # Call the function
    compute_and_save_embeddings(
        texts,
        doc_ids,
        model_name='all-MiniLM-L6-v2',
        batch_size=8,
        out_emb_path=output_emb_file,
        out_ids_path=output_ids_file,
        overwrite=True
    )
    print("Computation finished and files saved.")

# Loading and verification (Load from the specific paths defined above)
print("-" * 30)
print("Verifying loaded data...")

loaded_emb = np.load(output_emb_file, mmap_mode='r')
with open(output_ids_file, 'r', encoding='utf-8') as f:
    loaded_ids = json.load(f)

print("Embeddings shape:", loaded_emb.shape)
print("Number of doc ids:", len(loaded_ids))
print("Example embedding (first doc) first 10 dims:", loaded_emb[0][:10])

Output files not found. Starting data loading and embedding computation...


Embedding batches: 100%|██████████| 6240/6240 [20:56<00:00,  4.97it/s]

Saved embeddings -> data/processed/embeddings.npy
Saved doc ids -> data/processed/doc_ids.json
Computation finished and files saved.
------------------------------
Verifying loaded data...
Embeddings shape: (49919, 384)
Number of doc ids: 49919
Example embedding (first doc) first 10 dims: [-0.08914597  0.03614483 -0.02356093  0.02534425 -0.03705267 -0.02898317
  0.07373413 -0.0226588   0.00625514 -0.05217345]


A FAISS index is used.

In [17]:
def build_faiss_index(embeddings_path='data/processed/embeddings.npy', index_path='data/processed/faiss.index',
                      index_type='hnsw', ef_construction=200, M=32):
    emb = np.load(embeddings_path, mmap_mode='r')  # shape (N, d)
    d = emb.shape[1]
    if index_type == 'flat':
        index = faiss.IndexFlatIP(d)  # inner product -> cosine if vectors normalized
        index.add(emb)
    elif index_type == 'hnsw':
        index = faiss.IndexHNSWFlat(d, M)  # M controls connectivity
        index.hnsw.efConstruction = ef_construction
        index.add(emb)
    else:
        raise ValueError('index_type not supported')
    faiss.write_index(index, index_path)
    return index

In [18]:
def retrieve_similar_articles(query, model, embeddings, articles, top_n=5, use_ann=False):
    query_emb = model.encode([query], convert_to_numpy=True)
    # normalize
    query_emb = query_emb / np.linalg.norm(query_emb, axis=1, keepdims=True)
    
    if use_ann:
        index = build_faiss_index()
        distances, indices = index.search(query_emb.astype('float32'), top_n)
    else:
        # Exact search using sklearn
        nbrs = NearestNeighbors(n_neighbors=top_n, metric="cosine").fit(embeddings)
        distances, indices = nbrs.kneighbors(query_emb)
    
    results = pd.DataFrame({
        "article_id": [articles[i] for i in indices[0]],
        "similarity": [1 - d for d in distances[0]]
    })
    return results

In [ ]:
def get_representative(member_id, file_path="data/processed/communities.json", top_n=1):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)


        for group in data:
            n = min(top_n, len(group.get("community", [])))
            if member_id in group.get("community", []):
                return group["community"][n]
                
        return None

    except (FileNotFoundError, json.JSONDecodeError):
        return None

In [ ]:
query  ="I am a scientist"


top_n = 1

embeddings = np.load('data/processed/embeddings.npy', mmap_mode='r')
results = retrieve_similar_articles(query, model, embeddings, doc_ids, top_n, use_ann=True)

print('QUERY :')
print(query)
print("---")

print(f"Closest article")
print(article_name[results['article_id'][0]])
print("----")

print("Represenrative article")
representative = get_representative(results['article_id'][0])
print(article_name[representative])


QUERY :
I am a scientist
---
Closest article
Crowdsourced science: sociotechnical epistemology in the e-research
  paradigm
----
Represenrative article
Crowdsourced science: sociotechnical epistemology in the e-research
  paradigm


For evaluation purposes, it may be relevant to see if the response proposed by the method is at least in the same cluster as the one we were aiming for. This allows for a more refined approach than simply saying whether the article is correct or not. This is exactly what the function below does. (It even gives the minimum distance between articles in case they are related (otherwise None) -> not yet i i am not even sure it is interesting...).

In [ ]:
def common_commu(result_id, target_id):
    # are the result_id and the tardget_id in the same community / cluster ?
    COM_FILE = 'data/processed/communities.json'
    with open(COM_FILE, 'r') as file:
        commu = json.load(file)
    
    for commu_l in commu:
        if result_id in commu_l['community'] and target_id in commu_l['community']:
            return True
    
    return False

# Method: Locality Sensitive Hashing (LSH)

## Imports

In [57]:
import mmh3
import json 
from typing import Dict, Any
import numpy as np
from tqdm import tqdm
from mmh3 import hash

## Utilitary functions

### Find the index of a given element in a list

In [14]:
def find_index(L, x):
    """
    Calculates the index (i) of the first occurrence of element x in list L.

    Args:
        L (list): The list to search within.
        x (any): The element whose index is being sought.

    Returns:
        int: The index of element x in L.

    Raises:
        ValueError: If element x is not found in list L.
    """
    try:
        # The index() method returns the index of the first occurrence
        # of the specified element.
        index = L.index(x)
        return index
    except ValueError:
        # index() raises a ValueError if the element is not found.
        # It's good practice to handle this error.
        raise ValueError(f"The element '{x}' is not in the list.")

### Function to obtain a shingle list from an input text with a shingle size of q

In [4]:
punctuation = ['.',',',';',':','"']


def string_modulo_q(q, caracter_list, beginning_index):
    """takes a caracter list and a beginning index and returns a string composed of 
     the caracters in order beginning at the beginning index """
    assert q == len(caracter_list)

    string=""

    for i in range(q):
        string += caracter_list[(beginning_index+i)%q]

    return string


def shingle(q, text, punctuation_list = punctuation):
    "Returns the set of q-shingles from the original text"
    n = len(text)
    assert(n>q), "Text too short or shingle too long"
    assert q>=2, "Shingle size must be > 1 caracter "

    ind = 0 # index of the first caracter
    q_list = ["" for _ in range(q)]
    S = []
    ind_modified = 0

    for c in text:
        if (not (c in punctuation_list)) & (c != ' ') : # We ignore punctuation
            
            # Update q_string with another caracter
            q_list[ind] = c
            # Add one to the beginning index
            new_ind = (ind+1)%q
            ind = new_ind
            ind_modified += 1
            # New shingle added to the list
            S.append(string_modulo_q(q = q, caracter_list = q_list, beginning_index = ind))
    return S[q-1:]

### Function minhash to perform minhashing singature with size k

In [ ]:
# hashes a list of strings
def listhash(l,seed):
	val = 0
	for e in l:
		val = val ^ hash(e, seed)
	return val 

# Minhash function
def minhash(shingle_list, k):
    "returns a list of k different minhashes of the shingle list"
    return [min(listhash(s, seed) for s in shingle_list) for seed in range(k)]


### Function signatures to compute the signature matrix of a list of articles 

article = dictionnary of the form {"id" : ..., "clean_text": ..., ...}

In [6]:
def signatures(doc_list, shingle_size, signature_size):
    """
    inputs :
        - doc_lists : list of document of the form {'id': ... , 'abstract' : ... }
        - signature_size : size of the signatures
    outputs :
        - sig : signature matrix, every column represent the signature of a document
        - idx_to_sig : dictionnary that matches idexes (columns of sig) withs ids of the documents
    """
    idx_to_id = {} # dictionary of the signatures of each document
    n = len(doc_list)
    sig = np.zeros((signature_size,n))
    # signature of doc no "id" using minhashing on the shingle_list. Size of the shingles is shingle_size
    for i in tqdm(range(n), desc="Computing signatures"):
        id = doc_list[i]["id"]
        abstract = doc_list[i]["clean_text"]
        idx_to_id[i] = id
        sig[:,i] = np.array(minhash(shingle(q=shingle_size, text=abstract), k=signature_size))
    print("min hashing of the documents complete")
    return sig, idx_to_id

### Function band_hash to hash bands parts of the signature matrix in the LSH process

In [7]:
def lsh_band_hash(band, m, lsh_seed) -> int:
    """
    Computes a hash value for a single band (a list of r integers).
    The goal is to map identical bands to the same hash bucket.

    band: A list of r integers representing the signature's portion 
            for a specific band.
    Returns: An integer hash value for the band.
    """
    
    band_string = ",".join(map(str, band)) # Convert the band to a string representation.
    
    hash_value = mmh3.hash(band_string, lsh_seed) # Compute the hash using MurmurHash3.
    
    return abs(hash_value) % m # Ensure non-negative and fit within m buckets

In [8]:
def Jaccard_similarity_signatures(input_signature, doc_signature) -> float :
    "Returns an approximation of the jaccard similarity between 2 documents doc_name1 and doc_name2 using signatures"
    sig = doc_signature 
    S = 0
    k = len(sig) # size of signature list
    assert k == len(input_signature), "Signatures are not matching size"
    for i in range(k): # Loop over the signature pairs from the two documents
        if sig[i]==input_signature[i]:
            S+=1
    return S/k

def Jaccard_similarity_shingles(shingles_list_A, shingles_list_B):
    """
    Calculates the Jaccard similarity coefficient between two lists of shingles.

    Args:
        shingles_list_A (list): The list of shingles for the first document.
        shingles_list_B (list): The list of shingles for the second document.

    Returns:
        float: The Jaccard similarity score (0.0 to 1.0).
    """

    # 1. Convert lists to sets for efficient set operations and to ensure uniqueness
    set_A = set(shingles_list_A)
    set_B = set(shingles_list_B)

    # 2. Calculate the size of the intersection (common shingles)
    intersection_size = len(set_A.intersection(set_B))
    
    # Alternatively: intersection_size = len(set_A & set_B)

    # 3. Calculate the size of the union (all unique shingles combined)
    union_size = len(set_A.union(set_B))
    
    # Alternatively: union_size = len(set_A | set_B)

    # 4. Calculate the Jaccard score
    if union_size == 0:
        # Avoid division by zero if both lists are empty
        return 0.0

    jaccard_score = intersection_size / union_size
    return jaccard_score


def Jaccard_similarity(input_text ,article_list, candidate_idx, q):
    return Jaccard_similarity_shingles(
        shingles_list_A= shingle(q = q, text = input_text),
        shingles_list_B= shingle(q = q, text = article_list[candidate_idx]["clean_text"]))

### Functions Jaccard_similarity to calculate the Jaccard similarity between documents (shingle lists)

In [9]:
def Jaccard_similarity_shingles(shingles_list_A, shingles_list_B):
    """
    Calculates the Jaccard similarity coefficient between two lists of shingles.

    Args:
        shingles_list_A (list): The list of shingles for the first document.
        shingles_list_B (list): The list of shingles for the second document.

    Returns:
        float: The Jaccard similarity score (0.0 to 1.0).
    """

    # 1. Convert lists to sets for efficient set operations and to ensure uniqueness
    set_A = set(shingles_list_A)
    set_B = set(shingles_list_B)

    # 2. Calculate the size of the intersection (common shingles)
    intersection_size = len(set_A.intersection(set_B))
    
    # Alternatively: intersection_size = len(set_A & set_B)

    # 3. Calculate the size of the union (all unique shingles combined)
    union_size = len(set_A.union(set_B))
    
    # Alternatively: union_size = len(set_A | set_B)

    # 4. Calculate the Jaccard score
    if union_size == 0:
        # Avoid division by zero if both lists are empty
        return 0.0

    jaccard_score = intersection_size / union_size
    return jaccard_score


def Jaccard_similarity(input_text ,article_list, candidate_idx, q):
    return Jaccard_similarity_shingles(
        shingles_list_A= shingle(q = q, text = input_text),
        shingles_list_B= shingle(q = q, text = article_list[candidate_idx]["clean_text"]))

### Function lsh to return the most relevant articles using LSH method

preprocess_lsh gives a simplified dataset from the path of an original dataset to keep only the keys "id" and "clean_text" for every article

In [80]:
# Preprocessing the dataset to keep only the relevant information

def preprocess_lsh(dataset_path):
    """
    Input : 
        dataset_path : path of the json dataset of articles 
    Output :
        article_list : list of dictionnaries of the form {'id': ..., 'abstract': ...}  
    """
    try : 
        with open(dataset_path, 'r', encoding='utf-8') as f:
            data: Dict[str, Any] = json.load(f)
        print(f"Data succesfully loaded")
        
        article_list = [{'id': article['id'], 'clean_text' : article['clean_text']} for article in data] 
        return article_list

    #data = {'articles' : [d1 = {'id' : ..., 'authors' : ...,'abstract': ..., 'clean_text' : ... , 'categories' : ... , 'refs' :  ... } , d2, ...]}

    except Exception as e:
        print(f"Error in loading of the dataset : {e}")


In [86]:

# Implementing lsh function         

def lsh(input, article_list ,signature_matrix, idx_to_id, m, shingle_size, nb_band, band_size):
    """
    Inputs :
        input : input text from which we want to obtain sources
        article_list : list of arcticles i.e. dictionnaries of the format {'id': ..., 'abstract': ...}
        shingle_size : size of the shingle decomposition on which the minhashing is computed
        nb_band : number of horizontal bands in the signature matrix 
        band_size : number of rows per band in the signature matrix
        signature_size : size of the signatures of the documents obtained from minhashing of
                        the shingle size with signature_size different seeds. 
                        signature_size = nb_band*band_size

    Process :
        - Shingle all documents from the dataset and compute a signature for every document
          using minhashing (signatures function)
        - Find the documents that are most likely to be similar to input using LSH method
        - Compute the actual similarity between input and these document to eliminate false positives
    
    Outputs :
        - Most_similar : list of the most similar documents
        - Scores : list of Jaccard_similarities between input and documents 
    
    """

    # 1. Compute the a signature for every document

    print("Computing the signature of every document in the dataset ...")
    k = band_size*nb_band # Signature size

    # Compute signature of input
    print("Computing signature of input ...")
    input_signature = signatures([{'id':'input', 'clean_text':input}],
                                 shingle_size = shingle_size,
                                 signature_size = k)[0][:,0]
    
    # 2. Find the documents that are most likely to be similar to input using LSH method

    print("Performing LSH to find similar candidates ...")
    similar_candidates = {}
    n = len(signature_matrix[0])  # number articles in the dataset

    for band_nb in tqdm(range(nb_band), desc="LSH Bands"):
        input_hash = lsh_band_hash(
            band = input_signature[band_nb*band_size:(band_nb+1)*band_size],
            m = m,
            lsh_seed = band_nb
        )
        for i in range(n):
            doc_hash = lsh_band_hash(
                band = signature_matrix[:,i][band_nb*band_size:(band_nb+1)*band_size],
                m = m,
                lsh_seed = band_nb
                )
            if doc_hash == input_hash :
                if i in similar_candidates :
                    similar_candidates[i] += 1
                else :
                    similar_candidates[i] = 1
    print("LSH successfully performed to find similar candidates")

    # 3. Compute the actual similarity between input and these documents

    print("Calculation of the actual similarities ... \n")
    Ordered_similar_candidates = similar_candidates.keys()
    Ordered_similarities = []
    for idx in similar_candidates :
        j = Jaccard_similarity(input_text= input, 
                               article_list= article_list,
                               candidate_idx=idx,
                               q=shingle_size)
        Ordered_similarities.append(j)
    Most_similar = zip(Ordered_similar_candidates,Ordered_similarities)
    Most_similar = sorted(Most_similar, key = lambda pair:pair[1], reverse = True)
    Scores = [ p[1] for p in Most_similar]
    Most_similar = [idx_to_id[p[0]]['id'] for p in Most_similar]
    
    return Most_similar, Scores


### Function lsh_n to return the n most relevant document using lsh

In [66]:
def lsh_n(n,input, article_list ,signature_matrix, idx_to_id, m, shingle_size, nb_band, band_size) : 
    """ Returns the Most_similar list of lsh with the n most relevant results only. If the number of result from lsh 
    is < n, random articles from the dataset are added"""

    Most_similar = lsh(input, article_list ,signature_matrix, idx_to_id, m, shingle_size, nb_band, band_size)[0]
    length = len(Most_similar)
    if length < n :
        c = 0
        while length < n :
            extra_id = article_list[c]['id']
            if extra_id not in Most_similar :
                Most_similar.append(extra_id)
                length += 1
            c += 1    
    return Most_similar[:n]

## Choice of the dataset

In [89]:
datatypes = ["most_cited", "quartiles", "stratified"]
method = 1

## Preprocess

Keep ony the important part of the data

In [90]:
json_path = f"data/clean_subdataset_{datatypes[method]}.json" 
data = preprocess_lsh(dataset_path = json_path)
print("preprocessing done")

Data succesfully loaded
preprocessing done


In [91]:
print(data[0])

{'id': '1404.3723', 'clean_text': 'we highlight the progress current status and open challenges of qcddriven\nphysics in theory and in experiment we discuss how the strong interaction is\nintimately connected to a broad sweep of physical problems in settings ranging\nfrom astrophysics and cosmology to stronglycoupled complex systems in\nparticle and condensedmatter physics as well as to searches for physics\nbeyond the standard model we also discuss how success in describing the strong\ninteraction impacts other fields and in turn how such subjects can impact\nstudies of the strong interaction in the course of the work we offer a\nperspective on the many research streams which flow into and out of qcd as\nwell as a vision for future developments'}


Save the preprocessed data into a json file

In [92]:
output_json_path = f'data/subdataset_lsh_{datatypes[method]}.json'
with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4)

## Compute the signatures for everydataset

Load the simplified data previously saved

In [93]:
data_path = f'data/subdataset_lsh_{datatypes[method]}.json'
with open(data_path, 'r', encoding='utf-8') as f:
            data: Dict[str, Any] = json.load(f)

Compute the signature matrix :
- the shingle size is fixed to q = 7 caraters as it has been proven to be a good size to perform similarity calculations
- The cutting of the signature matrix in b = 10 bands of r = 10 rows gives good results in LSH method

In [ ]:
q = 7 
b = 10
r = 10

signature_matrix_Nmost, idx_to_id = signatures(
    doc_list=data,
    shingle_size = q,
    signature_size = b*r
    )

Computing signatures:  98%|█████████▊| 48845/49980 [06:28<00:13, 81.46it/s] 

Save the results

In [ ]:
np.save(file = f"data/signature_lsh_{datatypes[method]}_q{q}_b{b}_r{r}", arr = signature_matrix_Nmost)
output_json_path = f"data/idx_to_id_lsh_{datatypes[method]}_q{q}_b{b}_r{r}.json"
with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4)

## Perform the research of the most relevant documents using LSH

In [84]:
data_path = f'data/subdataset_lsh_{datatypes[method]}.json'
with open(data_path, 'r', encoding='utf-8') as f:
            data: Dict[str, Any] = json.load(f)

q = 7 
b = 10
r = 10

try :
    signature_matrix = np.load(f"data/signature_lsh_{datatypes[method]}_q{q}_b{b}_r{r}.npy")
    with open(f"data/idx_to_id_lsh_{datatypes[method]}_q{q}_b{b}_r{r}.json", 'r', encoding='utf-8') as f:
            idx_to_id: Dict[str, Any] = json.load(f)
except FileNotFoundError as e :
    print(f"No signature matrix has been saved with the set of parameters : q = {q}, b = {b}, r = {r} ")

test : Try to find the most relevant document with the first document of the dataset as input

In [87]:
Most_similar_100, Scores = lsh(
        input = data[0]['clean_text'],
        article_list=data,
        signature_matrix = signature_matrix,
        idx_to_id = idx_to_id,
        shingle_size=q,
        m = signature_matrix.shape[1]//10, 
        nb_band = b,
        band_size = r,
        )

print("Most similar documents : " , Most_similar_100, '\n')
print("Scores : " , Scores)

Computing the signature of every document in the dataset ...
Computing signature of input ...


Computing signatures: 100%|██████████| 1/1 [00:00<00:00,  7.72it/s]


min hashing of the documents complete
Performing LSH to find similar candidates ...


LSH Bands: 100%|██████████| 10/10 [00:02<00:00,  3.66it/s]


LSH successfully performed to find similar candidates
Calculation of the actual similarities ... 

Most similar documents :  ['0706.3095', '0809.3240', '0710.4166', '1105.2551', '1103.2598', '1005.2357', '0707.3419', '0707.4367', '1001.0569', '1011.2205', '1105.6335', '0811.4293', '0804.0252', '1012.4788', '1106.1172', '1003.5037', '0907.4901', '0901.2966', '1105.4714', '0710.5373', '0911.2664', '1009.5414', '0901.1847', '0808.0017', '0903.3467', '0806.2867', '0705.4311', '0709.0898', '0805.1746', '1011.2149', '1003.3387', '1003.1366', '0707.2100', '0803.0827', '1007.4001', '0805.3649', '1109.6641', '0903.5204', '1107.4942', '0810.4328', '0912.2441', '1008.3263', '1109.2329', '1105.6360', '0806.4044', '0811.4176', '1005.0617', '1112.0234', '0707.2559', '1109.2117', '1109.2314', '0908.0907', '0906.0003', '1112.6354', '0808.3629', '1008.3745', '0910.2702', '0808.3936', '1001.2937', '1003.2185', '0910.3803', '1010.4478', '1106.0522', '0906.1789', '0908.0538', '0906.1926', '0906.4547', '09

test : Try to find the n most relevant documents with the first document of the dataset as input

In [88]:
n=10
Most_similar_100_n = lsh_n(
    n,
    input = data[0]['clean_text'],
        article_list=data,
        signature_matrix = signature_matrix,
        idx_to_id = idx_to_id,
        m = signature_matrix.shape[1]//10, 
        shingle_size = q,
        nb_band = b,
        band_size = r,
)

print(f"{n} most similar documents : ", Most_similar_100_n )

Computing the signature of every document in the dataset ...
Computing signature of input ...


Computing signatures: 100%|██████████| 1/1 [00:00<00:00, 11.50it/s]


min hashing of the documents complete
Performing LSH to find similar candidates ...


LSH Bands: 100%|██████████| 10/10 [00:02<00:00,  4.42it/s]


LSH successfully performed to find similar candidates
Calculation of the actual similarities ... 

10 most similar documents :  ['0706.3095', '0809.3240', '0710.4166', '1105.2551', '1103.2598', '1005.2357', '0707.3419', '0707.4367', '1001.0569', '1011.2205']


Test on gold set


Calculates the Top-N Accuracy (Recall@N) for a specific recommendation method.

In [ ]:
def calculate_top_n_accuracy(method_name, n, gold_set_path, **kwargs):
    """
    Calculates Top-N Accuracy (Recall@N) adapting to the specific signatures 
    of Embedding, LSH, and Graph functions.

    Args:
        method_name (str): 'embedding', 'lsh', or 'graph'.
        n (int): Number of items to retrieve (k).
        gold_set_path (str): Path to the .json gold set file.
        **kwargs: Variable arguments required for the specific methods 
                  (e.g., model, embeddings, signature_matrix, etc.).

    Returns:
        float: The calculated accuracy.
    """
    
    # 1. Load the Gold Set
    try:
        with open(gold_set_path, 'r', encoding='utf-8') as f:
            gold_set = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Error loading gold set: {e}")
        return 0.0

    hit_count = 0
    total_samples = len(gold_set)
    
    print(f"--- Starting evaluation for method: {method_name} @ Top-{n} ---")

    # 2. Evaluation Loop
    for i, entry in tqdm(enumerate(gold_set), total=total_samples):
        
        true_id = entry.get('id_reference')
        query_text = entry.get('text')

        # Validation
        if not true_id or not query_text:
            continue

        recommended_ids = []
        # Embedding method
        if method_name == "embedding":
            # Requires: model, embeddings, articles (list of IDs)
            if not all(k in kwargs for k in ("model", "embeddings", "articles")):
                raise ValueError("Embedding method requires 'model', 'embeddings', and 'articles' in kwargs.")
            
            df_results = retrieve_similar_articles(
                query=query_text,
                model=kwargs['model'],
                embeddings=kwargs['embeddings'],
                articles=kwargs['articles'],
                top_n=n,
                use_ann=kwargs.get('use_ann', False)
            )
            
            # Extract IDs from the returned DataFrame
            recommended_ids = df_results['article_id'].tolist()

        # LSH method
        elif method_name == "LSH":
            # Requires: article_list, signature_matrix, idx_to_id, etc.
            required_lsh = ["article_list", "signature_matrix", "idx_to_id", "m", "shingle_size", "nb_band", "band_size"]
            if not all(k in kwargs for k in required_lsh):
                raise ValueError(f"LSH method requires: {required_lsh}")

            recommended_ids = lsh_n(
                n=n,
                input=query_text, 
                article_list=kwargs['article_list'],
                signature_matrix=kwargs['signature_matrix'],
                idx_to_id=kwargs['idx_to_id'],
                m=kwargs['m'],
                shingle_size=kwargs['shingle_size'],
                nb_band=kwargs['nb_band'],
                band_size=kwargs['band_size']
            )

        # Graph-based method
        elif method_name == "graph":
            # Requires: file_path (optional in your func, but good to pass)
            # WARNING: Your graph function takes an ID ('member_id'), but the gold set provides TEXT.
            # This adapter assumes you might be passing the text as an ID or that 
            # the logic needs to be adjusted. For now, we pass query_text as member_id.
            
            file_path = kwargs.get('file_path', "data/processed/communities.json")
            
            # Call the function
            # Your function returns a SINGLE item (string) or None, not a list.
            result = get_representative(
                member_id=query_text, # Potentially problematic: Text passed where ID expected
                file_path=file_path,
                top_n=n
            )
            
            if result is not None:
                recommended_ids = [result] # Wrap single result in a list
            else:
                recommended_ids = []

        else:
            raise ValueError(f"Unknown method: {method_name}")

        # 3. Check for Match (Hit)
        # Normalize to strings to ensure '123' == 123
        rec_ids_str = [str(x) for x in recommended_ids]
        
        if str(true_id) in rec_ids_str:
            hit_count += 1

    # 4. Final Calculation
    accuracy = hit_count / total_samples if total_samples > 0 else 0.0
    
    print(f"--- Evaluation Complete ---")
    print(f"Matches: {hit_count}/{total_samples}")
    print(f"Accuracy: {accuracy:.4f}")
    
    return accuracy